# 03. Dynamic Difficulty Adjustment (DDA) Comparative Analysis
**Project:** Adaptive Game Director (RL-Based DDA for ViZDoom)

This research notebook empirically investigates and benchmarks three dynamic regimes:
1. **Condition A: Fixed Difficulty Baselines** (Skill 1 [Easy], Skill 3 [Normal], Skill 5 [Hard]).
2. **Condition B: Rule-Based Dynamic Difficulty Adjustment** (3-tier heuristic Flow State controller).
3. **Condition C: DQN Reinforcement Learning AI Game Director** (Deep Q-Network with Flow State rewards and safety envelopes).

### Primary Hypotheses:
- **H1 (Flow Adherence):** The learned DQN Director will maintain human & simulated players inside Csikszentmihalyi's Flow Channel $[35, 75\text{ HP}]$ for a significantly higher duration than static or rule-based baselines.
- **H2 (Distress Mitigation):** The Director's safety guardrails prevent sudden frustration spikes ($<35\text{ HP}$) without causing stagnation or boredom ($>75\text{ HP}$).

In [ ]:
import os
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
print("DDA Analysis environment initialized.")

## 1. Theoretical Framework: Csikszentmihalyi's Flow State Channel

In [ ]:
# Theoretical Flow State Model Visualization
skill = np.linspace(0, 10, 200)
challenge_low = 0.75 * skill - 0.5
challenge_high = 1.25 * skill + 1.5

fig, ax = plt.subplots(figsize=(10, 6))
ax.fill_between(skill, np.maximum(0, challenge_low), challenge_high, color='#10b981', alpha=0.25, label='Flow Channel [Optimal Pacing: 35-75 HP]')
ax.fill_between(skill, challenge_high, 15, color='#ef4444', alpha=0.20, label='Anxiety / Distress Zone (<35 HP)')
ax.fill_between(skill, 0, np.maximum(0, challenge_low), color='#3b82f6', alpha=0.20, label='Boredom / Comfort Zone (>75 HP)')

ax.set_xlim(0, 10)
ax.set_ylim(0, 15)
ax.set_xlabel('Player Skill Level / Combat Competence', fontsize=12, fontweight='bold')
ax.set_ylabel('Environmental Challenge / Monster Intensity', fontsize=12, fontweight='bold')
ax.set_title('Csikszentmihalyi Flow Model for FPS Dynamic Difficulty Adjustment', fontsize=14, fontweight='bold')
ax.legend(loc='upper left', frameon=True, facecolor='white', framealpha=0.9)
ax.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

## 2. Load Empirical Benchmark Results

In [ ]:
benchmark_csv = PROJECT_ROOT / "results" / "evaluation_benchmark.csv"
summary_json = PROJECT_ROOT / "results" / "benchmark_summary.json"

if benchmark_csv.exists():
    df_bench = pd.read_csv(benchmark_csv)
    print(f"Successfully loaded {len(df_bench)} empirical trials across scenarios:")
    print(df_bench['scenario'].value_counts())
else:
    print("Benchmark CSV not found. Please run: python scripts/evaluate.py")
    df_bench = pd.DataFrame()

if summary_json.exists():
    with open(summary_json, 'r') as f:
        summary = json.load(f)
    print("\nBenchmark Summary Table:")
    print(pd.DataFrame(summary).T[['flow_channel_mean_pct', 'distress_mean_pct', 'mean_survival_sec', 'mean_kills']])
display(df_bench.head())

## 3. Comparative Visualizations: Flow Retention & Negative Affect

In [ ]:
if not df_bench.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Panel A: Flow Channel Retention by Condition
    sns.barplot(
        data=df_bench,
        x='condition',
        y='flow_channel_ratio',
        ax=axes[0],
        palette=['#3b82f6', '#f59e0b', '#10b981'],
        capsize=0.1,
        edgecolor='black'
    )
    axes[0].set_title('Flow Channel Retention Rate (%) - Higher is Better', fontweight='bold')
    axes[0].set_ylabel('Time in Flow Channel (%)')
    axes[0].set_ylim(0, 105)
    
    # Panel B: Distress Time Ratio by Condition
    sns.barplot(
        data=df_bench,
        x='condition',
        y='distress_ratio',
        ax=axes[1],
        palette=['#ef4444', '#f97316', '#a855f7'],
        capsize=0.1,
        edgecolor='black'
    )
    axes[1].set_title('Distress Time Ratio (<35 HP) - Lower is Better', fontweight='bold')
    axes[1].set_ylabel('Distress Time (%)')
    axes[1].set_ylim(0, 105)
    
    plt.tight_layout()
    plt.show()

## 4. Real-Time Telemetry Deep-Dive: Health Trajectory with AI Director Markers

In [ ]:
dqn_session_csv = PROJECT_ROOT / "data" / "telemetry" / "dqn_session.csv"
if dqn_session_csv.exists():
    session_df = pd.read_csv(dqn_session_csv)
    
    fig, ax1 = plt.subplots(figsize=(12, 6))
    
    # Flow channel bounds
    ax1.axhspan(35, 75, color='#10b981', alpha=0.15, label='Flow Channel [35, 75 HP]')
    ax1.axhspan(0, 35, color='#ef4444', alpha=0.1, label='Critical Distress (<35 HP)')
    
    # Health
    ax1.plot(session_df['elapsed_time_sec'], session_df['health'], color='#2563eb', linewidth=2.5, label='Health')
    ax1.set_xlabel('Elapsed Time (Seconds)', fontweight='bold')
    ax1.set_ylabel('Health (HP)', color='#2563eb', fontweight='bold')
    ax1.set_ylim(0, 105)
    
    # Ammo on secondary axis
    ax2 = ax1.twinx()
    ax2.plot(session_df['elapsed_time_sec'], session_df['ammo'], color='#f59e0b', linestyle='--', linewidth=1.8, label='Ammo')
    ax2.set_ylabel('Ammo Count', color='#f59e0b', fontweight='bold')
    
    # Interventions
    interventions = session_df[session_df['director_action'] != 'NO_CHANGE'].drop_duplicates(subset=['director_action', 'director_actuation'])
    if not interventions.empty:
        ax1.scatter(interventions['elapsed_time_sec'], interventions['health'], color='#dc2626', s=70, zorder=5, label='Director Action')
    
    plt.title('DQN Director Pacing: Combat Dynamics Over Time', fontsize=14, fontweight='bold')
    ax1.legend(loc='lower left', frameon=True)
    plt.tight_layout()
    plt.show()
else:
    print("No DQN session log found at data/telemetry/dqn_session.csv")